# Phase 4 Track A -- Tracked-cell, paired within-cell SMI comparison (DREADD saline/DCZ cohort)

Separate notebook from `4.SessionComparison.ipynb` (which is Track B only) so the tracked-cell analysis can be developed/re-run independently. Joins Phase 3's already-saved Track A output (`{group}_track_a_smi.csv`, one row per tracked cell with `SMI_<label>`/`valid_<label>`/`analysis_reliable_<label>` per session -- from `3.SMICalculation.py`'s `save_track_a_joined_table`) and runs the actual paired hypothesis test -- **no new SMI computation happens here**, and no cross-session cell tracking either (that's Phase 2). DCZ1/DCZ2/DCZ3 are run separately, never pooled, so you can see whether the effect replicates group by group.

Ported from `4.SessionComparison.py` (Functions 4.9-4.13, plus `discover_smi_sessions`/save utilities reused from Functions 4.2/4.8 to keep this notebook self-contained -- it doesn't depend on `4.SessionComparison.ipynb` having been run first).

Each function below gets its own test cell against real data (DCZ1, JSY093) right after its definition -- same style as `4.SessionComparison.ipynb`. The "loop it over every group" driver only shows up at the very end, once every piece has been checked individually.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # for plt.show() popups
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.stats import wilcoxon, ttest_rel

# Existing pipeline code, reused via import -- not modified. Needed to
# reload spatial_activity straight from preproc.h5 (see the trace-pulling
# cells below).
from helper import files

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Point this at whichever animal you're processing.
TEST_ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"


## Setup -- `discover_smi_sessions`

Reused unchanged from `4.SessionComparison.py`'s Function 4.2 -- needed here just to locate each session's `tseries_dir` (so each Track A group's `TrackedGroups` folder can be found from its saline/reference session).

In [ ]:
def discover_smi_sessions(animal_dir):
    """
    Scan animal_dir for every already-computed *_smi_results_dreadd.h5
    file, labeling each by whichever known naming pattern its TSeries
    folder matches. Same disambiguation behavior as discover_animal_sessions
    (Phase 3, Function 3.5) -- nothing is silently dropped on a label
    collision.

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'save_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'),
                                   recursive=True))

    entries = []  # (base_label, session_type, save_path, tseries_dir, tseries_name)
    unmatched = []

    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, save_path, tseries_dir, tseries_name))

    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, save_path, tseries_dir, tseries_name in entries:
        label = f'{base_label}__{tseries_name}' if label_counts[base_label] > 1 else base_label

        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['save_path']}, skipping {save_path}")
            continue

        catalog[label] = {
            'save_path': save_path,
            'session_type': session_type,
            'tseries_dir': tseries_dir,
        }

    print(f"Discovered {len(catalog)} sessions with saved SMI results under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['save_path']}")

    collided_labels = [l for l, c in label_counts.items() if c > 1]
    if collided_labels:
        print(f"\n{len(collided_labels)} label(s) had multiple TSeries and were disambiguated: "
              f"{collided_labels}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern "
              f"(labeled 'unknown'): {unmatched}")

    return catalog

In [ ]:
# --- Try it on the real animal dir ---
smi_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)

## Setup -- `discover_track_a_groups`

Auto-discovers this animal's Track A groups (DCZ1/DCZ2/DCZ3, ...) straight from `smi_catalog` + whatever `*_track_a_smi.csv` files Phase 3 already saved, instead of hardcoding `saline_label`/`dcz_label` strings per animal. The hardcoded version is exactly what's been silently wrong further down this notebook: every label embeds the animal's own ID (`..._JSY090_...` vs `..._JSY093_...`), so a `TRACK_A_GROUPS` config copied from one animal's run either `KeyError`s or (worse, if two animals happen to share a date) silently pairs the wrong sessions the moment `TEST_ANIMAL_DIR` changes. Needed now specifically to run the same paired comparison on JSY090 (null) and JSY093 (responder) without hand-maintaining two separate hardcoded configs.

## Setup -- save utilities

Reused unchanged from `4.SessionComparison.py`'s Function 4.8 -- needed by `save_track_a_group_outputs` (Function 4.13) later on.

In [ ]:
def save_dataframe_csv(df, output_dir, filename):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=False)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path


def _json_safe(obj):
    """Recursively convert numpy scalar types to native Python for json.dump."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(data, output_dir, filename):
    """
    Save a JSON-serializable dict to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    with open(save_path, 'w') as f:
        json.dump(_json_safe(data), f, indent=2)
    print(f"Saved -> {save_path}")
    return save_path

## Function 4.9 -- `load_track_a_smi_table`

In [ ]:
def load_track_a_smi_table(tracked_groups_dir, group_name):
    """
    Reload one Phase 3 Track A group's joined per-cell SMI table, saved by
    3.SMICalculation.py's save_track_a_joined_table as
    '{group_name}_track_a_smi.csv'. Just a pd.read_csv -- reimplemented
    here rather than imported (digit-prefixed module names aren't cleanly
    importable, same convention used throughout this pipeline).

    Parameters
    ----------
    tracked_groups_dir : str
        The TrackedGroups folder Phase 2/3 saved this group into.
    group_name : str

    Returns
    -------
    joined_df : pandas.DataFrame
    """
    table_path = os.path.join(tracked_groups_dir, f'{group_name}_track_a_smi.csv')
    joined_df = pd.read_csv(table_path)
    print(f"Loaded Track A joined table <- {table_path} ({len(joined_df)} tracked cells)")
    return joined_df

In [ ]:
# One real DREADD tracked-cell group's already-saved Track A output, to
# develop and sanity-check each function against as we go.
TEST_GROUP_NAME = 'DCZ1'
TEST_SALINE_LABEL = '260724_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE'
TEST_DCZ_LABEL = '260724_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ' 

# --- Try it on the real DCZ1 group's already-saved Track A output ---
test_tracked_groups_dir = os.path.join(smi_catalog[TEST_SALINE_LABEL]['tseries_dir'], 'TrackedGroups')
print(f'smi_catalog[TEST_SALINE_LABEL]: {smi_catalog[TEST_SALINE_LABEL]}')
print(test_tracked_groups_dir)

test_joined_df = load_track_a_smi_table(test_tracked_groups_dir, TEST_GROUP_NAME)

test_joined_df.head(10)

### Diagnostic -- how much do `valid` and `analysis_reliable` actually overlap?

Before assuming you need BOTH `valid_<label>` AND `analysis_reliable_<label>` in BOTH sessions (the current filter in `compare_smi_paired_track_a`), check how much N each criterion is actually costing you, and how much they overlap -- if one is nearly redundant with the other, dropping it recovers real N without loosening anything meaningful.

In [ ]:
def diagnose_valid_vs_reliable_overlap(joined_df, saline_label, dcz_label):
    """
    Cheap check before assuming you need BOTH valid_<label> AND
    analysis_reliable_<label> in both sessions: how much do these two
    filters actually overlap within each session, and how much N would
    each relaxed alternative recover for the both-sessions filter?

    Parameters
    ----------
    joined_df : pandas.DataFrame
        From load_track_a_smi_table.
    saline_label, dcz_label : str

    Returns
    -------
    summary : dict
        Counts for every filter combination printed below.
    """
    valid_s = joined_df[f'valid_{saline_label}']
    valid_d = joined_df[f'valid_{dcz_label}']
    rel_s = joined_df[f'analysis_reliable_{saline_label}']
    rel_d = joined_df[f'analysis_reliable_{dcz_label}']
    n_total = len(joined_df)

    def report(label, mask):
        print(f"  {label}: {int(mask.sum())}/{n_total}")

    print(f"n_total tracked cells: {n_total}\n")

    print("Per-session, per-criterion:")
    report("valid (saline)", valid_s)
    report("valid (dcz)", valid_d)
    report("analysis_reliable (saline)", rel_s)
    report("analysis_reliable (dcz)", rel_d)

    print("\nOverlap within saline session (valid vs analysis_reliable):")
    report("valid & analysis_reliable", valid_s & rel_s)
    report("valid & NOT analysis_reliable", valid_s & ~rel_s)
    report("NOT valid & analysis_reliable", ~valid_s & rel_s)

    print("\nOverlap within dcz session (valid vs analysis_reliable):")
    report("valid & analysis_reliable", valid_d & rel_d)
    report("valid & NOT analysis_reliable", valid_d & ~rel_d)
    report("NOT valid & analysis_reliable", ~valid_d & rel_d)

    print("\nResulting N under different both-sessions filter choices:")
    current = valid_s & valid_d & rel_s & rel_d
    valid_only = valid_s & valid_d
    reliable_only = rel_s & rel_d
    either = (valid_s | rel_s) & (valid_d | rel_d)

    report("current (valid AND analysis_reliable, both sessions)", current)
    report("valid only, both sessions", valid_only)
    report("analysis_reliable only, both sessions", reliable_only)
    report("(valid OR analysis_reliable), both sessions", either)

    return {
        'n_total': n_total,
        'current': int(current.sum()),
        'valid_only': int(valid_only.sum()),
        'reliable_only': int(reliable_only.sum()),
        'either': int(either.sum()),
    }

In [ ]:
# --- Try it on the real DCZ1 joined table ---
overlap_summary = diagnose_valid_vs_reliable_overlap(test_joined_df, TEST_SALINE_LABEL, TEST_DCZ_LABEL)

### Pull each tracked cell's spatial tuning curve (trial x bin), per session

Not from Phase 3's saved `*_smi_results_dreadd.h5` -- its `bin_centers` is internally rescaled for its own curve fitting (same caveat Phase 6's landmark work already flagged). This reloads `spatial_activity` straight from each session's own `preproc.h5`, then indexes it with each tracked cell's `roi_idx_<label>` (Phase 2's tracking-assigned position in that session) -- no new computation, just extraction. `spatial_activity` is (n_cells, n_trials, n_bins); pulling one cell out gives you its (n_trials, n_bins) tuning curve for that session.

In [ ]:
def load_tracked_cell_spatial_traces(joined_df, session_catalog, labels=None):
    """
    Pull every tracked cell's spatial_activity (raw trial x bin tuning
    curve) AND norm_spatial_activity (Preprocess.py's own per-lap,
    per-cell min/max-normalized version -- RT.normalize_spatial_activity,
    the same field helper/ResponseVisualization.py's response/waterfall
    plots and this group's own 3.SMICalculation.py already consume) from
    every session's own preproc.h5.

    Parameters
    ----------
    joined_df : pandas.DataFrame
        From load_track_a_smi_table -- needs 'global_cell_id' and
        roi_idx_<label> columns (Phase 2's build_master_cell_table).
    session_catalog : dict
        From discover_smi_sessions (or Phase 3's discover_animal_sessions)
        -- needs 'tseries_dir' per label present in joined_df.
    labels : list of str, optional
        Which roi_idx_<label> columns to pull -- defaults to every label
        present in joined_df (i.e. every session in this tracking group).

    Returns
    -------
    traces_by_cell : dict
        {global_cell_id: {label: (n_trials, n_bins) array}} -- raw
        spatial_activity. A label is only present for a cell if that
        session's preproc.h5 was found.
    norm_traces_by_cell : dict
        Same shape as traces_by_cell, but norm_spatial_activity -- each
        lap already min/max-normalized to [0, 1] by Preprocess.py, not
        renormalized here.
    bin_centers_by_label : dict
        {label: (n_bins,) array} -- raw cm-scale bin centers, per session
        (can differ slightly session to session).
    """
    roi_idx_cols = [c for c in joined_df.columns if c.startswith('roi_idx_')]
    all_labels = [c[len('roi_idx_'):] for c in roi_idx_cols]
    labels = labels if labels is not None else all_labels

    spatial_activity_by_label = {}
    norm_spatial_activity_by_label = {}
    bin_centers_by_label = {}
    for label in labels:
        if label not in session_catalog:
            print(f"WARNING: '{label}' not in session_catalog -- skipping.")
            continue
        tseries_dir = session_catalog[label]['tseries_dir']
        preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
        if not preproc_files:
            print(f"WARNING: no *preproc*.h5 found in {tseries_dir} for '{label}' -- skipping.")
            continue
        preproc_data = files.read_h5(preproc_files[0])
        spatial_activity_by_label[label] = preproc_data['spatial_activity']
        norm_spatial_activity_by_label[label] = preproc_data['norm_spatial_activity']
        bin_centers_by_label[label] = preproc_data['bin_centers']
        print(f"Loaded spatial_activity for '{label}': shape {preproc_data['spatial_activity'].shape}")

    traces_by_cell = {}
    norm_traces_by_cell = {}
    for _, row in joined_df.iterrows():
        cell_id = int(row['global_cell_id'])
        traces_by_cell[cell_id] = {}
        norm_traces_by_cell[cell_id] = {}
        for label in labels:
            if label not in spatial_activity_by_label:
                continue
            roi_idx = int(row[f'roi_idx_{label}'])
            traces_by_cell[cell_id][label] = spatial_activity_by_label[label][roi_idx]
            norm_traces_by_cell[cell_id][label] = norm_spatial_activity_by_label[label][roi_idx]

    print(f"\nPulled spatial traces for {len(traces_by_cell)} tracked cells across "
          f"{len(spatial_activity_by_label)}/{len(labels)} session(s).")

    return traces_by_cell, norm_traces_by_cell, bin_centers_by_label

In [ ]:
# --- Try it on the real DCZ1 tracked cells ---
test_traces_by_cell, test_norm_traces_by_cell, test_bin_centers_by_label = load_tracked_cell_spatial_traces(
    test_joined_df, smi_catalog
)

# Peek at one cell's traces
example_cell_id = test_joined_df['global_cell_id'].iloc[0]
for label, trace in test_traces_by_cell[example_cell_id].items():
    norm_trace = test_norm_traces_by_cell[example_cell_id][label]
    print(f"cell {example_cell_id}, '{label}': raw trace shape {trace.shape}, "
          f"norm trace shape {norm_trace.shape}")

### All tracked cells, saline vs. dcz -- heatmap sorted by peak position, marked by reliability/validity

Every tracked cell shown (NOT filtered to valid/analysis_reliable), one panel per session, peak-normalized and sorted by each cell's own peak position -- same spirit as `helper/ResponseVisualization.py`'s `create_response_plot` (peak-sorted heatmap), but reimplemented here since that function drops non-reliable cells entirely and this is deliberately about seeing where the reliable/valid subset sits *within* the full tracked population, not just the subset itself. A status strip alongside each heatmap marks each cell green (valid & analysis_reliable), orange (one but not both), or gray (neither). The returned per-cell table also carries each cell's own peak position *and* its `SMI_<label>` value side by side -- directly answers "what's the SMI where the peak is," per cell, per session.

In [ ]:
def plot_all_tracked_cell_traces_heatmap(traces_by_cell, bin_centers_by_label, joined_df, labels=None):
    """
    Heatmap of every tracked cell's own trial-averaged tuning curve
    (peak-normalized per cell, sorted by peak position), one panel per
    session -- NOT filtered to valid/reliable cells. A color strip
    alongside each panel marks each cell's valid_<label>/
    analysis_reliable_<label> status.

    Parameters
    ----------
    traces_by_cell : dict
        From load_tracked_cell_spatial_traces.
    bin_centers_by_label : dict
        From load_tracked_cell_spatial_traces.
    joined_df : pandas.DataFrame
        Needs 'global_cell_id', SMI_<label>, valid_<label>,
        analysis_reliable_<label>.
    labels : list of str, optional
        Defaults to every label present across traces_by_cell, ordered
        saline session(s) first then dcz (by 'SALINE'/'DCZ' substring --
        NOT plain alphabetical sort, which would put '..._DCZ' before
        '..._SALINE').

    Returns
    -------
    fig : matplotlib.figure.Figure
    sort_info_by_label : dict
        {label: DataFrame(global_cell_id, peak_position_cm, SMI, valid,
        analysis_reliable)}, sorted by peak_position_cm ascending.
    """
    if labels is None:
        all_labels = {l for cell_traces in traces_by_cell.values() for l in cell_traces.keys()}
        # Saline first, dcz second, anything else (e.g. baseline Day
        # sessions) last -- plain alphabetical sort would put '..._DCZ'
        # before '..._SALINE' since 'D' < 'S'.
        labels = sorted(all_labels, key=lambda l: (0 if 'SALINE' in l.upper()
                                                     else 1 if 'DCZ' in l.upper() else 2, l))

    n_panels = len(labels)
    fig, axes = plt.subplots(1, n_panels, figsize=(9 * n_panels, 9))
    axes = np.atleast_1d(axes)

    sort_info_by_label = {}
    for ax, label in zip(axes, labels):
        cell_ids, trial_avg_traces = [], []
        for cell_id, cell_traces in traces_by_cell.items():
            if label not in cell_traces:
                continue
            cell_ids.append(cell_id)
            trial_avg_traces.append(np.mean(cell_traces[label], axis=0))
        cell_ids = np.array(cell_ids)
        trial_avg_traces = np.array(trial_avg_traces)

        # Peak-normalize each cell's own trace to [0, 1] for display --
        # same spirit as ResponseVisualization.create_response_plot's
        # per-cell normalization, just without dropping non-reliable cells.
        row_min = trial_avg_traces.min(axis=1, keepdims=True)
        row_max = trial_avg_traces.max(axis=1, keepdims=True)
        row_range = np.where(row_max - row_min > 0, row_max - row_min, 1.0)
        normalized = (trial_avg_traces - row_min) / row_range

        bin_centers = bin_centers_by_label[label]
        peak_bin_idx = np.argmax(trial_avg_traces, axis=1)
        peak_position_cm = bin_centers[peak_bin_idx]

        sort_order = np.argsort(peak_position_cm)
        sorted_cell_ids = cell_ids[sort_order]
        sorted_normalized = normalized[sort_order]
        sorted_peak_position = peak_position_cm[sort_order]

        smi_col, valid_col, reliable_col = f'SMI_{label}', f'valid_{label}', f'analysis_reliable_{label}'
        lookup = joined_df.set_index('global_cell_id')[[smi_col, valid_col, reliable_col]]
        smi_values = lookup.loc[sorted_cell_ids, smi_col].to_numpy()
        valid_flags = lookup.loc[sorted_cell_ids, valid_col].to_numpy()
        reliable_flags = lookup.loc[sorted_cell_ids, reliable_col].to_numpy()

        extent = [bin_centers.min(), bin_centers.max(), len(sorted_cell_ids), 0]
        ax.imshow(sorted_normalized, aspect='auto', cmap='viridis', extent=extent)
        ax.set_xlabel('Position (cm)')
        ax.set_ylabel('Cell (sorted by peak position)')
        ax.set_title(f'{label}\n({len(sorted_cell_ids)} tracked cells, all shown)')

        # Status strip: green = valid & analysis_reliable, orange = one
        # only, gray = neither.
        status_code = np.where(valid_flags & reliable_flags, 0,
                       np.where(valid_flags | reliable_flags, 1, 2))
        strip_cmap = matplotlib.colors.ListedColormap(['#2ca02c', '#ff7f0e', '#d3d3d3'])
        strip_ax = ax.inset_axes([1.02, 0, 0.05, 1], transform=ax.transAxes)
        strip_ax.imshow(status_code[:, None], aspect='auto', cmap=strip_cmap, vmin=0, vmax=2,
                         extent=[0, 1, len(sorted_cell_ids), 0])
        strip_ax.set_xticks([])
        strip_ax.set_yticks([])
        strip_ax.set_title('status', fontsize=10)

        sort_info_by_label[label] = pd.DataFrame({
            'global_cell_id': sorted_cell_ids,
            'peak_position_cm': sorted_peak_position,
            'SMI': smi_values,
            'valid': valid_flags,
            'analysis_reliable': reliable_flags,
        })

    legend_elements = [
        plt.Rectangle((0, 0), 1, 1, color='#2ca02c', label='valid & analysis_reliable'),
        plt.Rectangle((0, 0), 1, 1, color='#ff7f0e', label='valid OR analysis_reliable (not both)'),
        plt.Rectangle((0, 0), 1, 1, color='#d3d3d3', label='neither'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle('All tracked cells -- trial-averaged tuning curve, sorted by peak position', fontsize=16)
    plt.tight_layout()
    return fig, sort_info_by_label

In [ ]:
# --- Try it on the real DCZ1 tracked cells ---
test_heatmap_fig, test_sort_info_by_label = plot_all_tracked_cell_traces_heatmap(
    test_traces_by_cell, test_bin_centers_by_label, test_joined_df
)
plt.show()

for label, sort_df in test_sort_info_by_label.items():
    print(f"\n--- '{label}' ---")
    print(sort_df.to_string(index=False))

### All tracked cells, saline vs. dcz -- even vs. odd response profiles, both sorted by EVEN-trial peak (cross-validated version of the heatmap above)

The heatmap above sorts/normalizes each session's panel using its own all-trial average -- so a "clean-looking place field" there could in principle just be reflecting noise that happens to average out nicely, since the same data picks the peak AND gets displayed. This version splits each cell's trials into EVEN (`np.arange(0, n_trials, 2)`) and ODD (`np.arange(1, n_trials, 2)`) -- same convention `helper/SMI_Calculation.py`'s `calculate_SMI_improved` uses for its training/testing split -- and shows **both halves**, one row per cell, same row order in both panels: **row order (peak position) AND normalization scale both come from the EVEN-trial profile only**, then that exact same order and scale is reused for the ODD panel. If a cell's odd-trial response actually lines up with the position/magnitude picked from its held-out even trials, its row looks similar in both panels; if it doesn't, that mismatch is now visible directly instead of being hidden by each panel getting to normalize/sort itself.

(Design choice worth flagging: the ODD panel is deliberately NOT re-normalized to its own [0, 1] -- it's scaled by the EVEN panel's min/max, so a real magnitude drop between the two halves shows up as a dimmer row rather than being rescaled away. Say if you'd rather each panel be normalized independently.)

In [ ]:
def plot_all_tracked_cell_traces_heatmap_odd_even(traces_by_cell, bin_centers_by_label, joined_df, labels=None):
    """
    Two panels per session -- EVEN-trial response profile and ODD-trial
    response profile -- both peak-normalized and sorted by the SAME row
    order: each cell's peak position from the EVEN-trial profile only
    ('even' = testing, same convention as calculate_SMI_improved's
    training/testing split). The odd panel inherits the even panel's row
    order AND normalization scale rather than sorting/scaling itself --
    that's what makes this a genuine cross-validated check instead of
    just two independent heatmaps.

    Parameters
    ----------
    traces_by_cell : dict
        From load_tracked_cell_spatial_traces -- needs the RAW (n_trials,
        n_bins) trace per cell/session (not pre-averaged).
    bin_centers_by_label : dict
        From load_tracked_cell_spatial_traces.
    joined_df : pandas.DataFrame
        Needs 'global_cell_id', SMI_<label>, valid_<label>,
        analysis_reliable_<label>.
    labels : list of str, optional
        Defaults to every label present across traces_by_cell, ordered
        saline session(s) first then dcz.

    Returns
    -------
    fig : matplotlib.figure.Figure
    sort_info_by_label : dict
        {label: DataFrame(global_cell_id, peak_position_cm_even, SMI,
        valid, analysis_reliable)}, sorted by peak_position_cm_even
        ascending -- the row order used in both panels.
    """
    if labels is None:
        all_labels = {l for cell_traces in traces_by_cell.values() for l in cell_traces.keys()}
        labels = sorted(all_labels, key=lambda l: (0 if 'SALINE' in l.upper()
                                                     else 1 if 'DCZ' in l.upper() else 2, l))

    n_cols = len(labels)
    fig, axes = plt.subplots(2, n_cols, figsize=(9 * n_cols, 18), squeeze=False)

    sort_info_by_label = {}
    for col, label in enumerate(labels):
        cell_ids, even_profiles, odd_profiles = [], [], []
        for cell_id, cell_traces in traces_by_cell.items():
            trace = cell_traces.get(label)
            if trace is None:
                continue
            n_trials = trace.shape[0]
            # Same convention as calculate_SMI_improved's testing/training
            # split (helper/SMI_Calculation.py): even trial-indices are
            # "testing", odd are "training".
            even_indices = np.arange(0, n_trials, 2)
            odd_indices = np.arange(1, n_trials, 2)
            if len(even_indices) == 0 or len(odd_indices) == 0:
                continue
            cell_ids.append(cell_id)
            even_profiles.append(np.mean(trace[even_indices], axis=0))
            odd_profiles.append(np.mean(trace[odd_indices], axis=0))
        cell_ids = np.array(cell_ids)
        even_profiles = np.array(even_profiles)
        odd_profiles = np.array(odd_profiles)

        bin_centers = bin_centers_by_label[label]

        # Row order AND normalization scale both come from the EVEN
        # profile only -- the odd panel is drawn in that same order/scale,
        # never sorted or rescaled by its own data.
        peak_bin_idx = np.argmax(even_profiles, axis=1)
        peak_position_cm = bin_centers[peak_bin_idx]
        sort_order = np.argsort(peak_position_cm)

        even_row_min = even_profiles.min(axis=1, keepdims=True)
        even_row_max = even_profiles.max(axis=1, keepdims=True)
        even_row_range = np.where(even_row_max - even_row_min > 0, even_row_max - even_row_min, 1.0)

        even_normalized = (even_profiles - even_row_min) / even_row_range
        odd_normalized = (odd_profiles - even_row_min) / even_row_range  # same scale as even, on purpose

        sorted_cell_ids = cell_ids[sort_order]
        sorted_even = even_normalized[sort_order]
        sorted_odd = odd_normalized[sort_order]
        sorted_peak_position = peak_position_cm[sort_order]

        smi_col, valid_col, reliable_col = f'SMI_{label}', f'valid_{label}', f'analysis_reliable_{label}'
        lookup = joined_df.set_index('global_cell_id')[[smi_col, valid_col, reliable_col]]
        smi_values = lookup.loc[sorted_cell_ids, smi_col].to_numpy()
        valid_flags = lookup.loc[sorted_cell_ids, valid_col].to_numpy()
        reliable_flags = lookup.loc[sorted_cell_ids, reliable_col].to_numpy()

        extent = [bin_centers.min(), bin_centers.max(), len(sorted_cell_ids), 0]
        status_code = np.where(valid_flags & reliable_flags, 0,
                       np.where(valid_flags | reliable_flags, 1, 2))
        strip_cmap = matplotlib.colors.ListedColormap(['#2ca02c', '#ff7f0e', '#d3d3d3'])

        panels = [(sorted_even, 'EVEN trials (row order + scale from this half)'),
                  (sorted_odd, 'ODD trials (SAME row order + scale as even, not its own)')]
        for row, (panel_data, panel_name) in enumerate(panels):
            ax = axes[row, col]
            # Odd panel shares the even panel's scale (not renormalized to
            # its own [0, 1]), so it can legitimately fall outside [0, 1]
            # -- clip only the colormap display, not the underlying values.
            ax.imshow(panel_data, aspect='auto', cmap='viridis', extent=extent, vmin=0, vmax=1)
            ax.set_xlabel('Position (cm)')
            ax.set_ylabel('Cell (sorted by EVEN-trial peak position)')
            ax.set_title(f'{label}\n{panel_name}\n({len(sorted_cell_ids)} tracked cells, all shown)')

            strip_ax = ax.inset_axes([1.02, 0, 0.05, 1], transform=ax.transAxes)
            strip_ax.imshow(status_code[:, None], aspect='auto', cmap=strip_cmap, vmin=0, vmax=2,
                             extent=[0, 1, len(sorted_cell_ids), 0])
            strip_ax.set_xticks([])
            strip_ax.set_yticks([])
            strip_ax.set_title('status', fontsize=10)

        sort_info_by_label[label] = pd.DataFrame({
            'global_cell_id': sorted_cell_ids,
            'peak_position_cm_even': sorted_peak_position,
            'SMI': smi_values,
            'valid': valid_flags,
            'analysis_reliable': reliable_flags,
        })

    legend_elements = [
        plt.Rectangle((0, 0), 1, 1, color='#2ca02c', label='valid & analysis_reliable'),
        plt.Rectangle((0, 0), 1, 1, color='#ff7f0e', label='valid OR analysis_reliable (not both)'),
        plt.Rectangle((0, 0), 1, 1, color='#d3d3d3', label='neither'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle('All tracked cells -- even vs. odd trial response profiles, both sorted by even-trial peak position',
                 fontsize=16)
    plt.tight_layout()
    return fig, sort_info_by_label

In [ ]:
# --- Try it on the real DCZ1 tracked cells ---
test_oddeven_fig, test_oddeven_sort_info = plot_all_tracked_cell_traces_heatmap_odd_even(
    test_traces_by_cell, test_bin_centers_by_label, test_joined_df
)
plt.show()

for label, sort_df in test_oddeven_sort_info.items():
    print(f"\n--- '{label}' (first 15 of {len(sort_df)}) ---")
    print(sort_df.head(15).to_string(index=False))

### All tracked cells, saline vs. dcz -- one row per cell instead of overlaid (page 1 of the eventual PDF)

Instead of overlaying every cell's trace into one shared axis per (session, cell-type) panel above, give each tracked cell its own row: saline in the LEFT column, dcz in the RIGHT column (same order used everywhere else in this notebook -- `saline_label`/`dcz_label` argument order, `TRACK_A_GROUPS` config, etc.). Each panel plots `norm_spatial_activity` directly -- Preprocess.py's own per-lap, per-cell min/max-normalized version of `spatial_activity` (`RT.normalize_spatial_activity`, the same field `helper/ResponseVisualization.py`'s response/waterfall plots and this group's own `3.SMICalculation.py` already consume) -- rather than renormalizing `spatial_activity` ourselves here. Every individual lap is drawn thin/light-gray, with the lap-average overlaid as a thicker line plus a shaded +/- SEM band, on top. A colored box is drawn directly on a panel's own axes: **green** if `valid_<label>` AND `analysis_reliable_<label>` in that session, **orange** if only one of the two, no highlight if neither -- so you see at a glance which *session* (not just whether the pair as a whole passed) drove a cell's inclusion.

This is meant to become one PDF (`matplotlib.backends.backend_pdf.PdfPages`) with ~5-10 cells per page, looping over every tracked cell in a group. For now this is just `plot_tracked_cells_page` run on the first page's worth of cells, to check the layout before wiring up pagination + the actual save -- no PDF is written yet.

**On cross-validation:** the average curve drawn here is the plain mean across ALL laps in that session -- not split into train/test halves. That's fine for the box *coloring*, which is genuinely cross-validated already and computed independently upstream (Phase 3, not here): `SMI_<label>`/`valid_<label>` come from `helper/SMI_Calculation.py`'s `calculate_SMI_improved`, which fits the tuning curve on ODD trials ("training") and evaluates the response on EVEN trials ("testing"); `analysis_reliable_<label>` comes from a separate even/odd split-half correlation test (`helper/ReliabilityTesting.py`'s `test_cell_reliability`). There's no peak-picking or sort step riding on the all-lap average shown here (unlike the heatmap's row order, which *is* derived from the same data it displays), so there's no double-dipping in what's plotted.

**Does having an SMI value mean the cell is reliable? No.** `valid_<label>` just means Phase 3's SMI curve-fitting pipeline succeeded for that cell in that session (found a peak in the allowed region, found a usable non-preferred position, Rp+Rn > 0 -- see `calculate_SMI_improved` in `helper/SMI_Calculation.py`). `analysis_reliable_<label>` is a completely separate, independently-computed check (`test_cell_reliability` in `helper/ReliabilityTesting.py`): whether the even/odd trial-split correlation clears a shuffle-based threshold. A cell can absolutely have a numeric `SMI_<label>` (curve fit succeeded, `valid=True`) while its response is noisy/unreliable (`analysis_reliable=False`) -- that's exactly the "orange" case above, and part of why the box distinguishes the two rather than only showing whether a cell was `valid`.

In [ ]:
def plot_tracked_cells_page(cell_ids, norm_traces_by_cell, bin_centers_by_label, joined_df,
                             saline_label, dcz_label, page_figsize=(8.5, 11)):
    """
    One page of the eventual tracked-cell PDF: one row per cell in
    cell_ids, saline in the LEFT column and dcz in the RIGHT column. Each
    panel plots norm_spatial_activity directly -- every individual lap
    (thin, light gray) with the lap-average overlaid as a thicker line
    plus a shaded +/- SEM band, on top -- no renormalization here,
    Preprocess.py already min/max-normalized each lap. A colored box is
    drawn on a panel's own axes to mark that session's
    valid_<label>/analysis_reliable_<label> status -- green if both,
    orange if exactly one, default (thin black) axes if neither.

    Parameters
    ----------
    cell_ids : sequence of int
        Which tracked cells (global_cell_id) go on this page, top to
        bottom, in the order given.
    norm_traces_by_cell : dict
        From load_tracked_cell_spatial_traces's norm_traces_by_cell
        return value (norm_spatial_activity, already per-lap normalized).
    bin_centers_by_label : dict
        From load_tracked_cell_spatial_traces.
    joined_df : pandas.DataFrame
        Needs 'global_cell_id', SMI_<label>, valid_<label>,
        analysis_reliable_<label> for saline_label/dcz_label.
    saline_label, dcz_label : str
        Explicit, not inferred/sorted -- saline is always the left
        column, dcz always the right, regardless of how the label
        strings would sort alphabetically.
    page_figsize : (float, float)
        Figure size in inches -- defaults to a portrait letter page,
        since this is meant to become one page of a saved PDF.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    lookup = joined_df.set_index('global_cell_id')
    n_cells = len(cell_ids)
    fig, axes = plt.subplots(n_cells, 2, figsize=page_figsize, squeeze=False)

    for row, cell_id in enumerate(cell_ids):
        cell_norm_traces = norm_traces_by_cell.get(cell_id, {})
        for col, label in enumerate((saline_label, dcz_label)):
            ax = axes[row, col]
            norm_trace = cell_norm_traces.get(label)

            if norm_trace is None:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes,
                        fontsize=9, color='gray')
                ax.set_xticks([])
                ax.set_yticks([])
            else:
                bin_centers = bin_centers_by_label[label]
                trial_avg = np.mean(norm_trace, axis=0)
                trial_sem = np.std(norm_trace, axis=0) / np.sqrt(norm_trace.shape[0])

                for single_trial in norm_trace:
                    ax.plot(bin_centers, single_trial, color='gray', alpha=0.25,
                             linewidth=0.6, zorder=1)
                ax.fill_between(bin_centers, trial_avg - trial_sem, trial_avg + trial_sem,
                                 color='black', alpha=0.2, zorder=2)
                ax.plot(bin_centers, trial_avg, color='black', linewidth=2.2, zorder=3)

                valid_flag = bool(lookup.loc[cell_id, f'valid_{label}'])
                reliable_flag = bool(lookup.loc[cell_id, f'analysis_reliable_{label}'])
                if valid_flag and reliable_flag:
                    box_color, box_lw = '#2ca02c', 3
                elif valid_flag or reliable_flag:
                    box_color, box_lw = '#ff7f0e', 3
                else:
                    box_color, box_lw = 'black', 0.8
                for spine in ax.spines.values():
                    spine.set_edgecolor(box_color)
                    spine.set_linewidth(box_lw)

                smi_val = lookup.loc[cell_id, f'SMI_{label}']
                ax.text(0.03, 0.95, f'SMI={smi_val:.2f}', transform=ax.transAxes, fontsize=7, va='top')
                ax.tick_params(labelsize=7)

            if row == 0:
                ax.set_title('Saline' if col == 0 else 'DCZ', fontsize=12, fontweight='bold')
            if row == n_cells - 1:
                ax.set_xlabel('Position (cm)', fontsize=8)
            if col == 0:
                ax.set_ylabel('Norm. activity', fontsize=8)
                ax.annotate(f'cell {cell_id}', xy=(-0.4, 0.5), xycoords='axes fraction',
                            ha='right', va='center', rotation=90, fontsize=9)

    legend_elements = [
        plt.Line2D([0], [0], color='#2ca02c', linewidth=3, label='valid & analysis_reliable'),
        plt.Line2D([0], [0], color='#ff7f0e', linewidth=3, label='valid OR analysis_reliable (not both)'),
        plt.Line2D([0], [0], color='black', linewidth=0.8, label='neither'),
        plt.Line2D([0], [0], color='gray', linewidth=0.6, alpha=0.5, label='individual laps'),
        plt.Line2D([0], [0], color='black', linewidth=2.2, label='lap average (+/- SEM band)'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=1, fontsize=7, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle('Tracked cells -- norm_spatial_activity, all laps + average, saline vs. dcz, '
                 'boxed by valid/analysis_reliable status', fontsize=11)
    plt.tight_layout()
    return fig

In [ ]:
# --- Try it on just the first page's worth of real DCZ1 tracked cells ---
CELLS_PER_PAGE = 8  # within the 5-10/page range -- final PDF loop will chunk by this
first_page_cell_ids = test_joined_df['global_cell_id'].to_numpy()[:CELLS_PER_PAGE]

test_page_fig = plot_tracked_cells_page(
    first_page_cell_ids, test_norm_traces_by_cell, test_bin_centers_by_label, test_joined_df,
    TEST_SALINE_LABEL, TEST_DCZ_LABEL
)
plt.show()

## Function 4.10 -- `compare_smi_paired_track_a`

Track A's version of Function 4.7's hypothesis test. Same tracked cell's SMI compared saline vs. dcz, restricted to cells `valid` (reliable_valid_cells -- reliable AND the SMI curve fit succeeded, the exact filter Function 4.7 uses) AND `analysis_reliable` in BOTH sessions.

In [ ]:
def compare_smi_paired_track_a(joined_df, saline_label, dcz_label):
    """
    Paired Wilcoxon signed-rank (primary -- matches Function 4.7's "SMI is
    bounded/non-normal, use non-parametric" reasoning; unlike Phase 5/6's
    n=5-group paired tests, n here -- tens to ~100 tracked cells -- is
    large enough for Wilcoxon to have real resolving power rather than
    floor out at a fixed minimum p) plus a paired t-test (secondary,
    magnitude-aware -- the same pairing that mattered in Phase 5/6).

    Parameters
    ----------
    joined_df : pandas.DataFrame
        From load_track_a_smi_table.
    saline_label, dcz_label : str
        Must match this group's actual roi_idx_<label>/SMI_<label> column
        suffixes.

    Returns
    -------
    result : dict or None
        None (with a printed message) if fewer than 2 cells are valid &
        analysis_reliable in both sessions. Otherwise: {'n', 'saline_label',
        'dcz_label', 'median_saline', 'median_dcz', 'mean_saline',
        'mean_dcz', 'median_diff', 'mean_diff', 'wilcoxon_stat',
        'wilcoxon_p', 'ttest_stat', 'ttest_p', 'filtered_df'}. median_diff/
        mean_diff are dcz - saline (negative = SMI dropped under DCZ).
    """
    valid_col_saline = f'valid_{saline_label}'
    valid_col_dcz = f'valid_{dcz_label}'
    reliable_col_saline = f'analysis_reliable_{saline_label}'
    reliable_col_dcz = f'analysis_reliable_{dcz_label}'
    smi_col_saline = f'SMI_{saline_label}'
    smi_col_dcz = f'SMI_{dcz_label}'

    required_cols = [valid_col_saline, valid_col_dcz, reliable_col_saline,
                      reliable_col_dcz, smi_col_saline, smi_col_dcz]
    missing = [c for c in required_cols if c not in joined_df.columns]
    if missing:
        raise KeyError(f"joined_df is missing {missing} -- check saline_label/dcz_label match "
                        f"this group's actual session labels, and that Phase 3's "
                        f"join_smi_to_master_table has been re-run since valid_<label> was added.")

    both_ok = (joined_df[valid_col_saline] & joined_df[valid_col_dcz] &
               joined_df[reliable_col_saline] & joined_df[reliable_col_dcz])
    filtered_df = joined_df[both_ok].copy()
    n = len(filtered_df)

    if n < 2:
        print(f"Only {n} cell(s) valid & analysis_reliable in BOTH sessions -- nothing to test.")
        return None

    saline_vals = filtered_df[smi_col_saline].to_numpy()
    dcz_vals = filtered_df[smi_col_dcz].to_numpy()

    wilcoxon_stat, wilcoxon_p = wilcoxon(saline_vals, dcz_vals)
    ttest_stat, ttest_p = ttest_rel(saline_vals, dcz_vals)

    result = {
        'n': n,
        'saline_label': saline_label,
        'dcz_label': dcz_label,
        'median_saline': float(np.median(saline_vals)),
        'median_dcz': float(np.median(dcz_vals)),
        'mean_saline': float(np.mean(saline_vals)),
        'mean_dcz': float(np.mean(dcz_vals)),
        'median_diff': float(np.median(dcz_vals - saline_vals)),
        'mean_diff': float(np.mean(dcz_vals - saline_vals)),
        'wilcoxon_stat': float(wilcoxon_stat),
        'wilcoxon_p': float(wilcoxon_p),
        'ttest_stat': float(ttest_stat),
        'ttest_p': float(ttest_p),
        'filtered_df': filtered_df,
    }

    print(f"Track A paired comparison ({saline_label} vs {dcz_label}): n={n}")
    print(f"  group medians: saline={result['median_saline']:.4f}, dcz={result['median_dcz']:.4f} "
          f"(difference of medians: {result['median_dcz'] - result['median_saline']:.4f} -- "
          f"NOT the same as the line below)")
    print(f"  median of each cell's own (dcz - saline) change: {result['median_diff']:.4f}   "
          f"[paired with Wilcoxon below]")
    print(f"  group means:   saline={result['mean_saline']:.4f}, dcz={result['mean_dcz']:.4f} "
          f"(mean of each cell's own change: {result['mean_diff']:.4f})   [paired with t-test below]")
    print(f"  Wilcoxon signed-rank: stat={wilcoxon_stat:.3f}, p={wilcoxon_p:.4f}")
    print(f"  Paired t-test:        stat={ttest_stat:.3f}, p={ttest_p:.4f}")

    return result

In [ ]:
# --- Try it on the real DCZ1 joined table ---
test_result = compare_smi_paired_track_a(test_joined_df, TEST_SALINE_LABEL, TEST_DCZ_LABEL)

## Function 4.11 -- `plot_smi_paired_track_a`

SMI_saline (x) vs. SMI_dcz (y) per tracked cell, y=x reference line, colored by direction of change -- fits this N (tens to ~100 cells) far better than the slope plots Phase 5/6 used for their n=5 comparison groups.

In [ ]:
def plot_smi_paired_track_a(result, title=''):
    """
    SMI_saline (x) vs. SMI_dcz (y) per tracked cell, y=x reference line,
    colored by direction of change. Fits this N (tens to ~100 cells) far
    better than the slope plots Phase 5/6 used for their n=5 comparison
    groups, which would be unreadable at this N.

    Parameters
    ----------
    result : dict
        From compare_smi_paired_track_a (needs 'filtered_df', 'n',
        'saline_label', 'dcz_label').
    title : str

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    filtered_df = result['filtered_df']
    smi_col_saline = f"SMI_{result['saline_label']}"
    smi_col_dcz = f"SMI_{result['dcz_label']}"

    saline_vals = filtered_df[smi_col_saline].to_numpy()
    dcz_vals = filtered_df[smi_col_dcz].to_numpy()
    decreased = dcz_vals < saline_vals

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(saline_vals[decreased], dcz_vals[decreased], color='tab:red', s=25, alpha=0.6,
               label=f'DCZ < saline (n={int(decreased.sum())})')
    ax.scatter(saline_vals[~decreased], dcz_vals[~decreased], color='tab:blue', s=25, alpha=0.6,
               label=f'DCZ >= saline (n={int((~decreased).sum())})')

    lo = min(saline_vals.min(), dcz_vals.min())
    hi = max(saline_vals.max(), dcz_vals.max())
    ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', alpha=0.7, label='y = x')

    ax.set_xlabel('SMI (saline)')
    ax.set_ylabel('SMI (dcz)')
    ax.set_title(title or f"Track A paired comparison (n={result['n']})")
    ax.legend(loc='best', fontsize=13)
    ax.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    return fig

In [ ]:
# --- Try it on the real DCZ1 result ---
test_fig = plot_smi_paired_track_a(test_result, title=f"{TEST_GROUP_NAME} (Track A, paired within-cell)")
plt.show()

## Function 4.12 -- `run_track_a_paired_comparison`

Loops Functions 4.9-4.11 over however many Track A groups you give it. DCZ1/DCZ2/DCZ3 are run separately here, never pooled -- same "don't collapse across days" design decision Track B's comparison groups already use.

`saline_label`/`dcz_label` per group must match `3.SMICalculation.py`'s `TRACK_A_GROUPS` `reference_label`/`session_labels` exactly.

In [ ]:
def run_track_a_paired_comparison(track_a_groups_config):
    """
    Loops Functions 4.9-4.11 over however many Track A groups you give it.

    Parameters
    ----------
    track_a_groups_config : list of dict
        Each: {'group_name', 'tracked_groups_dir', 'saline_label', 'dcz_label'}.

    Returns
    -------
    track_a_results : dict
        {group_name: compare_smi_paired_track_a(...) result, plus a 'fig'
        key} -- only for groups where the comparison actually ran (n >= 2).
    """
    track_a_results = {}
    for cfg in track_a_groups_config:
        print(f"\n{'='*90}\nTRACK A -- {cfg['group_name']}\n{'='*90}")
        joined_df = load_track_a_smi_table(cfg['tracked_groups_dir'], cfg['group_name'])
        result = compare_smi_paired_track_a(joined_df, cfg['saline_label'], cfg['dcz_label'])
        if result is not None:
            result['fig'] = plot_smi_paired_track_a(
                result, title=f"{cfg['group_name']} (Track A, paired within-cell)"
            )
            track_a_results[cfg['group_name']] = result
            plt.show()

    return track_a_results

In [ ]:
# --- Now run it across all your Track A groups (DCZ1/DCZ2/DCZ3) ---
TRACK_A_GROUPS = [
    {
        'group_name': 'DCZ1',
        'saline_label': '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE',
        'dcz_label': '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ',
    },
    {
        'group_name': 'DCZ2',
        'saline_label': '260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_SALINE',
        'dcz_label': '260726_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_2_DCZ',
    },
    {
        'group_name': 'DCZ3',
        'saline_label': '260728_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_3_SALINE',
        'dcz_label': '260728_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_3_DCZ',
    },
]
for cfg in TRACK_A_GROUPS:
    cfg['tracked_groups_dir'] = os.path.join(smi_catalog[cfg['saline_label']]['tseries_dir'], 'TrackedGroups')

track_a_results = run_track_a_paired_comparison(TRACK_A_GROUPS)

## Function 4.13 -- save everything Track A generates

In [ ]:
def save_track_a_group_outputs(output_dir, group_name, result):
    """
    Save one Track A group's paired-comparison outputs: the filtered
    per-cell table (CSV), a stats summary (JSON), and the figure (PNG).

    Parameters
    ----------
    output_dir : str
    group_name : str
    result : dict
        From compare_smi_paired_track_a (with 'fig' added by
        run_track_a_paired_comparison).

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {
        'table': save_dataframe_csv(result['filtered_df'], output_dir,
                                     f"{group_name}_track_a_paired_cells.csv"),
    }

    summary = {k: v for k, v in result.items() if k not in ('filtered_df', 'fig')}
    saved_paths['summary'] = save_json(summary, output_dir, f"{group_name}_track_a_paired_stats.json")

    fig = result.get('fig')
    if fig is not None:
        saved_paths['figure'] = save_figure_png(fig, output_dir, f"{group_name}_track_a_paired_plot.png")

    return saved_paths


def save_all_track_a_outputs(output_dir, track_a_results):
    """
    Loop save_track_a_group_outputs over every group in track_a_results.

    Parameters
    ----------
    output_dir : str
    track_a_results : dict
        {group_name: result with 'fig'}, from run_track_a_paired_comparison.

    Returns
    -------
    saved_paths_by_group : dict
    """
    saved_paths_by_group = {}
    for group_name, result in track_a_results.items():
        saved_paths_by_group[group_name] = save_track_a_group_outputs(output_dir, group_name, result)
    print(f"\nSaved Track A outputs for {len(saved_paths_by_group)} group(s) to {output_dir}")
    return saved_paths_by_group

In [ ]:
# --- Save everything Track A generated for this animal ---
OUTPUT_DIR = os.path.join(TEST_ANIMAL_DIR, 'Phase4_SessionComparison_Results')
TRACK_A_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'TrackA')
track_a_saved_paths = save_all_track_a_outputs(TRACK_A_OUTPUT_DIR, track_a_results)